## Lab 2: Personalize our agent by adding memory

### Overview

In Lab 1, you built a Financial Product Launch Agent that worked well for a single product manager in a local session. However, real-world product launch teams need to scale beyond a single user running in a local environment.

When we run an **Agent in Production**, we'll need:
- **Multi-User Support**: Handle multiple product managers simultaneously
- **Persistent Storage**: Save product launch conversations beyond session lifecycle
- **Long-Term Learning**: Extract product manager preferences and launch patterns
- **Cross-Session Continuity**: Remember product managers across different launch projects

**Workshop Progress:**
- **Lab 1 (Done)**: Create Agent Prototype - Build a functional product launch agent
- **Lab 2 (Current)**: Enhance with Memory - Add conversation context and personalization
- **Lab 3**: Scale with Gateway & Identity - Share tools across agents securely
- **Lab 4**: Deploy to Production - Use AgentCore Runtime with observability
- **Lab 5**: Build User Interface - Create a product manager-facing application


In this lab, you'll add the missing persistence and learning layer that transforms your Goldfish-Agent (forgets the conversation in seconds) into a smart personalized Product Launch Assistant.

Memory is a critical component of intelligence. While Large Language Models (LLMs) have impressive capabilities, they lack persistent memory across conversations. [Amazon Bedrock AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-getting-started.html) addresses this limitation by providing a managed service that enables AI agents to maintain context over time, remember important facts, and deliver consistent, personalized experiences.

AgentCore Memory operates on two levels:
- **Short-Term Memory**: Immediate conversation context and session-based information that provides continuity within a single product launch interaction.
- **Long-Term Memory**: Persistent information extracted and stored across multiple product launches, including market insights, launch preferences, and patterns that enable personalized experiences over time.



### Prerequisites

* **AWS Account** with appropriate permissions
* **Python 3.10+** installed locally
* **AWS CLI configured** with credentials
* **Anthropic Claude 3.7** enabled on [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
* **Tavily API Key** from https://tavily.com
* **Strands Agents** and other libraries installed in the next cells

### Step 1: Import Libraries

Let's import the libraries for AgentCore Memory. For it, we will use the [Amazon Bedrock AgentCore Python SDK](https://github.com/aws/bedrock-agentcore-sdk-python), a lightweight wrapper that helps you working with AgentCore capabilities.

In [ ]:
import logging

# Import agentCore Memory
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

import boto3
from boto3.session import Session

boto_session = Session()
REGION = boto_session.region_name

logger = logging.getLogger(__name__)

from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter

### Step 2: Create Bedrock AgentCore Memory resources

Amazon Bedrock AgentCore Memory is a fully managed service that provides persistent memory capabilities for AI agents.

#### AgentCore Memory Concepts:

1. **Short-Term Memory (STM)**: Immediately stores conversation context within the session
2. **Long-Term Memory (LTM)**: Asynchronously processes STM to extract meaningful patterns, preferences and facts
3. **Memory Strategies**: Different approaches for extracting and organizing information:
   - **USER_PREFERENCE**: Learns product manager preferences, launch patterns, and working styles
   - **SEMANTIC**: Stores factual information about products, markets, and launches using vector embeddings for similarity search
4. **Namespaces**: Logical grouping of memories by product manager and context type. We'll create these two namespaces:
- `product-launch/pm/{actorId}/preferences`: Product manager preferences and launch patterns
- `product-launch/pm/{actorId}/semantic`: Factual information about products, markets, and previous launches

This structure enables multi-tenant memory where each product manager's information is isolated and easily retrievable.

#### Memory Creation Process:

Creating memory resources involves provisioning the underlying infrastructure (vector databases, processing pipelines, etc.). This typically takes 2-3 minutes as AWS sets up the managed services behind the scenes.

In [ ]:
memory_client = MemoryClient(region_name=REGION)
memory_name = "ProductLaunchMemory"

def create_or_get_memory_resource():
    try:
        memory_id = get_ssm_parameter("/app/productlaunch/agentcore/memory_id")
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        return memory_id
    except:
        try:
            strategies = [
                {
                    StrategyType.USER_PREFERENCE.value: {
                        "name": "PMPreferences",
                        "description": "Captures product manager preferences, launch patterns, and working styles",
                        "namespaces": ["product-launch/pm/{actorId}/preferences"],
                    }
                },
                {
                    StrategyType.SEMANTIC.value: {
                        "name": "ProductLaunchSemantic",
                        "description": "Stores facts about products, markets, and previous launches",
                        "namespaces": ["product-launch/pm/{actorId}/semantic"],
                    }
                },
            ]
            print("Creating AgentCore Memory resources. This will take 2-3 minutes...")
            print("While we wait, let's understand what's happening behind the scenes:")
            print("• Setting up managed vector databases for semantic search")
            print("• Configuring memory extraction pipelines")
            print("• Provisioning secure, multi-tenant storage")
            print("• Establishing namespace isolation for product manager data")
            # *** AGENTCORE MEMORY USAGE *** - Create memory resource with semantic strategy
            response = memory_client.create_memory_and_wait(
                name=memory_name,
                description="Financial product launch agent memory",
                strategies=strategies,
                event_expiry_days=90,          # Memories expire after 90 days
            )
            memory_id = response["id"]
            try:
                put_ssm_parameter("/app/productlaunch/agentcore/memory_id", memory_id)
            except:
                raise
            return memory_id
        except Exception as e:
            print(f"Failed to create memory resource: {e}")
            return None

In [ ]:
memory_id = create_or_get_memory_resource()
if memory_id:
    print("✅ AgentCore Memory created successfully!")
    print(f"Memory ID: {memory_id}")
else:
    print("Memory resource not created. Try Again !")

## Step 3: Seed previous product manager interactions

**Why are we seeding memory?**

In production, agents accumulate memory naturally through product manager interactions. However, for this lab, we're seeding historical product launch conversations to demonstrate how Long-Term Memory (LTM) works without waiting for real conversations.

**How memory processing works:**
1. `create_event` stores interactions in **Short-Term Memory** (STM) instantly
2. STM is asynchronously processed by **Long-Term Memory** strategies
3. LTM extracts patterns, preferences, and facts for future retrieval

Let's seed some product manager history to see this in action:

In [ ]:
# List existing memory resources
for memory in memory_client.list_memories():
    print(f"Memory Arn: {memory.get('arn')}")
    print(f"Memory ID: {memory.get('id')}")
    print("--------------------------------------------------------------------")

# Seed with previous product manager interactions
PM_ACTOR_ID = "pm_sarah_001"

previous_interactions = [
    ("We just launched our new auto loan product at 5.99% APR. Can you help me track the competitive positioning?","USER"),
    ("Congratulations on the auto loan launch at 5.99% APR! That's well-positioned against major banks at 6.25% and credit unions at 5.75%. I'll help you monitor the competitive landscape and track performance metrics for this launch.", "ASSISTANT"),
    ("I'm targeting millennials for our personal loan product. We decided on 3.99% APR. Can you create marketing materials?", "USER"),
    ("Perfect! I'll create marketing materials for your millennial-focused personal loan at 3.99% APR. I've generated a modern marketing poster with design elements that appeal to younger demographics, emphasizing fast approval and digital-first experience. The materials are ready for your review.", "ASSISTANT"),
    ("For our credit card launch, we need to understand compliance. What regulations apply?", "USER"),
    ("For your credit card launch, you'll need to comply with the CARD Act, TILA, FCRA, and ECOA at the federal level. State banking regulations and PCI DSS compliance are also required. The compliance timeline is typically 4-6 weeks. I recommend starting with legal team review and documentation preparation.", "ASSISTANT"),
]

# Save previous interactions
if memory_id:
    try:
        memory_client.create_event(
            memory_id=memory_id,
            actor_id=PM_ACTOR_ID,
            session_id="previous_session",
            messages=previous_interactions
        )
        print("✅ Seeded product manager history successfully")
        print("📝 Interactions saved to Short-Term Memory")
        print("⏳ Long-Term Memory processing will begin automatically...")
    except Exception as e:
        print(f"⚠️ Error seeding history: {e}")

### Understanding Memory Processing

After creating events with `create_event`, AgentCore Memory processes the data in two stages:

1. **Immediate**: Messages stored in Short-Term Memory (STM)
2. **Asynchronous**: STM processed into Long-Term Memory (LTM) strategies

LTM processing typically takes 20-30 seconds as the system:
- Analyzes conversation patterns
- Extracts product manager preferences and launch patterns
- Creates semantic embeddings for factual information about products and markets
- Organizes memories by namespace for efficient retrieval

Let's check if our Long-Term Memory processing is complete by retrieving product manager preferences:

In [ ]:
import time

# Wait for Long-Term Memory processing to complete
print("🔍 Checking for processed Long-Term Memories...")
retries = 0
max_retries = 6  # 1 minute wait

while retries < max_retries:
    memories = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"product-launch/pm/{PM_ACTOR_ID}/preferences",
        query="what are the product manager's launch preferences and patterns"
    )
    
    if memories:
        print(f"✅ Found {len(memories)} preference memories after {retries * 10} seconds!")
        break
    
    retries += 1
    if retries < max_retries:
        print(f"⏳ Still processing... waiting 10 more seconds (attempt {retries}/{max_retries})")
        time.sleep(10)
    else:
        print("⚠️ Memory processing is taking longer than expected. This can happen with overloading..")
        break

print("🎯 AgentCore Memory automatically extracted these product manager preferences from our seeded conversations:")
print("=" * 80)

for i, memory in enumerate(memories, 1):
    if isinstance(memory, dict):
        content = memory.get('content', {})
        if isinstance(content, dict):
            text = content.get('text', '')
            print(f"  {i}. {text}")

### Exploring Semantic Memory

Semantic memory stores factual information from conversations using vector embeddings. This enables similarity-based retrieval of relevant facts and context.

In [ ]:
import time
# Retrieve semantic memories (factual information)
while True:
    semantic_memories = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"product-launch/pm/{PM_ACTOR_ID}/semantic",
        query="information about product launches and market research"
    )
    print("🧠 AgentCore Memory identified these factual details from conversations:")
    print("=" * 80)
    if memories:
        break
    time.sleep(10)
for i, memory in enumerate(semantic_memories, 1):
    if isinstance(memory, dict):
        content = memory.get('content', {})
        if isinstance(content, dict):
            text = content.get('text', '')
            print(f"  {i}. {text}")

## Step 4: Implement Strands Hooks to save and retrieve agent interactions

Now we'll integrate AgentCore Memory with our agent using Strands' hook system. This creates an automatic memory layer that works seamlessly with any agent conversation.

- **MessageAddedEvent**: Triggered when messages are added to the conversation, allowing us to retrieve and inject product manager context
- **AfterInvocationEvent**: Fired after agent responses, enabling automatic storage of product launch interactions to memory

The hook system ensures memory operations happen automatically without manual intervention, creating a seamless experience where product manager context is preserved across product launches.

To create the hooks we will extend the `HookProvider` class:


In [ ]:
class ProductLaunchMemoryHooks(HookProvider):
    """Memory hooks for product launch agent"""

    def __init__(
        self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str
    ):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_pm_context(self, event: MessageAddedEvent):
        """Retrieve product manager context before processing launch query"""
        messages = event.agent.messages
        if (
            messages[-1]["role"] == "user"
            and "toolResult" not in messages[-1]["content"][0]
        ):
            user_query = messages[-1]["content"][0]["text"]

            try:
                all_context = []

                for context_type, namespace in self.namespaces.items():
                    # *** AGENTCORE MEMORY USAGE *** - Retrieve product manager context from each namespace
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,
                    )
                    # Post-processing: Format memories into context strings
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(
                                        f"[{context_type.upper()}] {text}"
                                    )

                # Inject product manager context into the query
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0][
                        "text"
                    ] = f"Product Manager Context:\n{context_text}\n\n{original_text}"
                    logger.info(f"Retrieved {len(all_context)} product manager context items")

            except Exception as e:
                logger.error(f"Failed to retrieve product manager context: {e}")

    def save_launch_interaction(self, event: AfterInvocationEvent):
        """Save product launch interaction after agent response"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last product manager query and agent response
                pm_query = None
                agent_response = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif (
                        msg["role"] == "user"
                        and not pm_query
                        and "toolResult" not in msg["content"][0]
                    ):
                        pm_query = msg["content"][0]["text"]
                        break

                if pm_query and agent_response:
                    # *** AGENTCORE MEMORY USAGE *** - Save the product launch interaction
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[
                            (pm_query, "USER"),
                            (agent_response, "ASSISTANT"),
                        ],
                    )
                    logger.info("Saved product launch interaction to memory")

        except Exception as e:
            logger.error(f"Failed to save product launch interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register product launch memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_pm_context)
        registry.add_callback(AfterInvocationEvent, self.save_launch_interaction)
        logger.info("Product launch memory hooks registered")



## Step 5: Create a Product Launch Agent with memory

Next, we will implement the Financial Product Launch Agent just as we did in Lab 1, but this time we instantiate the class `ProductLaunchMemoryHooks` and we pass the memory hook to the agent constructor.

In [ ]:
import uuid
import sys
import os

# Add labs directory to path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
if not current_dir.endswith('labs'):
    labs_path = os.path.join(current_dir, 'labs')
    if os.path.exists(labs_path) and labs_path not in sys.path:
        sys.path.insert(0, labs_path)

from strands import Agent
from strands.models import BedrockModel

# Import tools from Lab 1 (now using unified market research)
from lab_helpers.unified_market_research import market_research
from lab_helpers.enhanced_tools import browse_web
from lab_helpers.marketing_tools import create_marketing_poster

SESSION_ID = str(uuid.uuid4())
memory_hooks = ProductLaunchMemoryHooks(memory_id, memory_client, PM_ACTOR_ID, SESSION_ID)

SYSTEM_PROMPT = """You are an expert financial product launch assistant helping product managers launch new financial products.
Your role is to:
- Provide real-time market insights using intelligent market research
- Analyze competitive landscape with actual web data
- Support go-to-market strategy development
- Generate marketing materials
- Remember product manager preferences and previous launches
- Be professional, data-driven, and strategic in your recommendations

You have access to:
1. market_research() - Intelligent market research (auto-chooses quick or deep analysis)
2. browse_web() - Browse specific websites
3. create_marketing_poster() - Generate marketing materials

Always use these tools to get accurate, up-to-date information rather than making assumptions.
When you receive Product Manager Context at the start of a query, use it to personalize your response."""

# Initialize the Bedrock model (Anthropic Claude 4.5 Haiku)
model = BedrockModel(
    model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    temperature=0.3,
    region_name=REGION
)

# Create the product launch agent with memory hooks and unified market research
agent = Agent(
    model=model,
    hooks=[memory_hooks], # Pass Memory Hooks
    tools=[
        market_research,             # Unified intelligent market research
        browse_web,                  # Web browsing capability
        create_marketing_poster      # Marketing material generation
    ],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Financial Product Launch Agent with Memory and Unified Market Research created!")

## Step 6: Test Personalized Agent

Let's test our memory-enhanced agent! Watch how it uses the product manager's historical preferences and previous launches to provide personalized recommendations.

The agent will automatically:
1. Retrieve relevant product manager context from memory
2. Use that context to personalize the response based on previous launches
3. Save this new interaction for future use

### 🔍 What to Look For:

**Memory is working correctly if:**
- ✅ Agent answers immediately with specific details (5.99% APR, millennials, etc.)
- ✅ **NO tool calls** are made (no `research_market_data`, `browse_web`, etc.)
- ✅ Agent references "previous" conversations or launches
- ✅ Answers match the seeded data exactly

**Memory is NOT working if:**
- ❌ You see tool calls like `Tool #1: research_market_data`
- ❌ Agent says "Let me research..." or "I'll browse..."
- ❌ Generic answers without specific rates or details

Run the cells below and observe the agent's behavior:

In [ ]:
from IPython.display import display, Markdown

print("🚗 Test 1: Checking if agent remembers previous auto loan launch...")
print("Expected: Agent should recall that we previously researched auto loans at 5.99% APR")
print("="*80)
response1 = agent("What rate did I use for my last auto loan product launch?")
print("\n" + "="*80)
print("✅ Test 1 Complete - Check if the agent answered from memory without calling tools!")

In [ ]:
print("\n" + "="*80)
print("💳 Test 2: Checking if agent remembers my target demographic preferences...")
print("Expected: Agent should recall millennial-focused personal loan at 3.99% APR")
print("="*80)
response2 = agent("What demographic did I target in my personal loan launch, and what rate did I use?")
print("\n" + "="*80)
print("✅ Test 2 Complete - Check if the agent answered from memory without calling tools!")

print("\n" + "="*80)
print("📋 Test 3: Checking if agent remembers compliance work I've done...")
print("Expected: Agent should recall credit card compliance requirements (CARD Act, TILA, FCRA, ECOA, PCI DSS)")
print("="*80)
response3 = agent("I'm planning another credit card launch. What compliance requirements did we discuss before?")
print("\n" + "="*80)
print("✅ Test 3 Complete - Check if the agent answered from memory without calling tools!")

### Understanding What Just Happened

Notice how the Agent **directly answered from memory** without calling any tools:

**Test 1 Results:**
• ✅ Recalled your previous auto loan launch at 5.99% APR
• ✅ No market research tools were called - it used memory!

**Test 2 Results:**
• ✅ Remembered millennial-focused personal loan at 3.99% APR
• ✅ Retrieved this from Long-Term Memory preferences

**Test 3 Results:**
• ✅ Recalled credit card compliance requirements (CARD Act, TILA, FCRA, ECOA, PCI DSS)
• ✅ Retrieved from semantic memory of previous conversations

**Key Insight:** The agent answered these questions **instantly from memory** rather than doing new market research. This is the power of AgentCore Memory - persistent, personalized product launch experiences!

### Memory vs. Real-Time Research

Compare this to Lab 1 where the agent would have called multiple tools (research_market_data, browse_web, etc.) for every question. With memory:
- **Faster responses** - no API calls needed for known information
- **Consistent recommendations** - remembers your preferences
- **Personalized experience** - adapts to your launch patterns

## Congratulations! 🎉

You have successfully completed **Lab 2: Add memory to the Financial Product Launch Agent**!

### What You Accomplished:

- Created a serverless managed memory with Amazon Bedrock AgentCore Memory
- Implemented long-term memory to store Product Manager Preferences and Semantic (Factual) information about launches
- Integrated AgentCore Memory with the Financial Product Launch Agent using the hook mechanism provided by Strands Agents
- Tested personalized responses based on previous product launch history

##### Next Up [Lab 3 - Scaling with Gateway and Identity  →](lab-03-add-gateway.ipynb)

## Resources
- [Amazon Bedrock Agent Core Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)
- [Amazon Bedrock AgentCore Memory Deep Dive blog](https://aws.amazon.com/blogs/machine-learning/amazon-bedrock-agentcore-memory-building-context-aware-agents/)
- [Strands Agents Hooks Documentation](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/hooks/?h=hooks)